In [ ]:
# 1. Configuracion

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "si7006_t2"
TBL_SILVER   = f"{CATALOG}.{SCHEMA}.orders_silver"
TBL_GOLD_KPI = f"{CATALOG}.{SCHEMA}.gold_kpi_ventana"
VOL_CHECKPOINTS = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"
CKPT_GOLD_KPI = f"{VOL_CHECKPOINTS}/gold_kpi_ventana"

WINDOW_SIZE = "1 minute"
WATERMARK_DELAY = "30 seconds"

print(f"Silver:       {TBL_SILVER}")
print(f"Gold KPI:     {TBL_GOLD_KPI}")
print(f"Checkpoint:   {CKPT_GOLD_KPI}")
print(f"Window size:  {WINDOW_SIZE}")
print(f"Watermark:    {WATERMARK_DELAY}")


In [ ]:
# 2. Lectura streaming desde Silver + agregacion por ventana

df_gold_kpi = (
    spark.readStream
        .format("delta")
        .table(TBL_SILVER)
        .withWatermark("event_time", WATERMARK_DELAY)
        .groupBy(
            F.window("event_time", WINDOW_SIZE).alias("w"),
            F.col("country"),
            F.col("is_guest")
        )
        .agg(
            F.approx_count_distinct("invoice_no").alias("n_orders"),
            F.count("*").alias("n_lines"),
            F.round(F.sum("total_amount"), 2).alias("revenue"),
            F.sum("quantity").alias("units_sold"),
            F.approx_count_distinct("customer_id").alias("n_distinct_customers")
        )
        .withColumn(
            "avg_ticket",
            F.when(
                F.col("n_orders") > 0,
                F.round(F.col("revenue") / F.col("n_orders"), 2)
            ).otherwise(F.lit(None).cast("double"))
        )
        .select(
            F.col("w.start").alias("window_start"),
            F.col("w.end").alias("window_end"),
            "country",
            "is_guest",
            "n_orders",
            "n_lines",
            "revenue",
            "units_sold",
            "avg_ticket",
            "n_distinct_customers"
        )
)

df_gold_kpi.printSchema()


In [ ]:
# 3. Loop de escritura con Trigger.AvailableNow

import time

ITERACIONES = 5
PAUSA_SEGUNDOS = 12

for i in range(1, ITERACIONES + 1):
    query = (
        df_gold_kpi.writeStream
            .format("delta")
            .outputMode("append")
            .option("checkpointLocation", CKPT_GOLD_KPI)
            .option("mergeSchema", "false")
            .trigger(availableNow=True)
            .toTable(TBL_GOLD_KPI)
    )
    query.awaitTermination()

    total = spark.sql(f"SELECT COUNT(*) AS n FROM {TBL_GOLD_KPI}").collect()[0]["n"]
    print(f"[{i:02d}/{ITERACIONES}] {time.strftime('%H:%M:%S')} | total: {total:,}")

    if i < ITERACIONES:
        time.sleep(PAUSA_SEGUNDOS)


In [ ]:
# 4. Validacion rapida

spark.table(TBL_GOLD_KPI).printSchema()

spark.sql(f"""
    SELECT COUNT(*) AS filas,
           COUNT(DISTINCT window_start) AS ventanas,
           ROUND(SUM(revenue), 2) AS revenue_total,
           SUM(n_orders) AS orders_total,
           SUM(n_lines) AS lines_total,
           SUM(units_sold) AS units_total
    FROM {TBL_GOLD_KPI}
""").show(truncate=False)

spark.sql(f"SELECT * FROM {TBL_GOLD_KPI} ORDER BY window_start DESC, country ASC LIMIT 20").show(truncate=False)


In [ ]:
# 5. Historial Delta

spark.sql(f"DESCRIBE HISTORY {TBL_GOLD_KPI}").select(
    "version", "timestamp", "operation",
    "operationMetrics.numOutputRows"
).show(truncate=False)
